In [ ]:
# RAILPULSE | NB_02_Bronze_To_Silver  (lakehouses attachés : LH_RailPulse_Bronze et LH_RailPulse_Silver)
# ---- Cellule 1 : contrôle qualité des circulations, quarantaine, MERGE Delta ----
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
fr = lambda n: f"{n:,}".replace(",", " ")   # 98 950 plutôt que 98,950

BRONZE = "LH_RailPulse_Bronze"
SILVER = "LH_RailPulse_Silver"
try:   # chemin OneLake complet : marche quel que soit le lakehouse par défaut
    LANDING = notebookutils.lakehouse.getWithProperties(BRONZE)["properties"]["abfsPath"] + "/Files/landing"
except Exception:
    LANDING = "Files/landing"

raw = spark.read.table(f"{BRONZE}.bronze_trips")
n_raw = raw.count()
dedup = raw.dropDuplicates()
codes = [r.station_code for r in spark.read.table(f"{BRONZE}.bronze_stations").select("station_code").collect()]

typed = dedup.withColumn("arrival_delay_min", F.expr("try_cast(arrival_delay_min as int)"))
checked = typed.withColumn("dq_reason",
    F.when(~F.col("origin").isin(codes) | ~F.col("destination").isin(codes), "GARE_INCONNUE")
     .when(F.col("arrival_delay_min") < -30, "RETARD_IMPOSSIBLE"))

quarantine = checked.filter("dq_reason IS NOT NULL").withColumn("_loaded_at", F.current_timestamp())
quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_quarantine")

latest = Window.partitionBy("trip_id").orderBy(F.col("scheduled_departure").desc())
clean = (checked.filter("dq_reason IS NULL")
         .withColumn("rn", F.row_number().over(latest)).filter("rn = 1"))

trips = (clean.select(
            "trip_id",
            F.to_date("service_date").alias("service_date"),
            "line_code", "origin", "destination",
            F.to_timestamp("scheduled_departure").alias("scheduled_departure"),
            F.to_timestamp("scheduled_arrival").alias("scheduled_arrival"),
            "trainset_id",
            (F.col("status") == "CANCELLED").alias("is_cancelled"),
            "arrival_delay_min",
            F.expr("try_cast(seats as int)").alias("seats"),
            F.expr("try_cast(distance_km as int)").alias("distance_km"))
         .withColumn("dep_hour", F.hour("scheduled_departure"))
         .withColumn("is_peak", F.col("dep_hour").isin(7, 8, 17, 18).cast("int"))
         .withColumn("_loaded_at", F.current_timestamp()))

target = f"{SILVER}.silver_trips"
if spark.catalog.tableExists(target):
    (DeltaTable.forName(spark, target).alias("t")
        .merge(trips.alias("s"), "t.trip_id = s.trip_id")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    trips.write.format("delta").saveAsTable(target)

n_silver = spark.read.table(target).count()
print(f"Bronze : {fr(n_raw)} | après dédoublonnage : {fr(dedup.count())} | quarantaine : {fr(quarantine.count())} | Silver : {fr(n_silver)}")

In [ ]:
# ---- Cellule 2 : réservations, rames, horaire à prévoir ----
bookings = (spark.read.parquet(f"{LANDING}/bookings/*.parquet")
            .withColumn("_source_file", F.col("_metadata.file_name"))
            .withColumn("_ingested_at", F.current_timestamp()))
bookings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE}.bronze_bookings")

valid_ids = spark.read.table(target).select("trip_id")
silver_bk = (spark.read.table(f"{BRONZE}.bronze_bookings").join(valid_ids, "trip_id", "left_semi")
             .select("trip_id", F.col("travel_class").cast("int"), "channel",
                     F.col("passengers").cast("int"), F.col("revenue_eur").cast("decimal(18,2)"),
                     F.col("avg_lead_days").cast("double")))
silver_bk.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_bookings")

(spark.read.option("multiLine", True).json(f"{LANDING}/reference/trainsets.json")
    .withColumn("trainset_age", F.year(F.current_date()) - F.col("commissioning_year"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_trainsets"))

(spark.read.option("header", True).option("inferSchema", True).csv(f"{LANDING}/plan/timetable_next14.csv")
    .withColumn("service_date", F.to_date("service_date"))
    .withColumn("scheduled_departure", F.to_timestamp("scheduled_departure"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_timetable_plan"))

n_bk_raw = spark.read.table(f"{BRONZE}.bronze_bookings").count()
n_bk = spark.read.table(f"{SILVER}.silver_bookings").count()
print(f"Réservations Bronze : {fr(n_bk_raw)} | Silver : {fr(n_bk)} | orphelines écartées : {fr(n_bk_raw - n_bk)}")

In [ ]:
# ---- Cellule 3 : compactage V-Order (lecture Direct Lake plus rapide) ----
for t in ["silver_trips", "silver_bookings"]:
    spark.sql(f"OPTIMIZE {SILVER}.{t} VORDER")
print("Silver prêt.")